# 08: Find a low-energy quantum state

**Level:** Intermediate  
**Before you start:** Notebook 03; vectors and matrices.  
**Resources:** CPU unless an optional remote step is enabled.

Can a trainable circuit approach the lowest energy of a two-spin system?

Run each cell in order. All core calculations are written in this notebook.

## 1. Define the physical question

A Hamiltonian assigns energy to a state. We use H = -Z₀Z₁ - 0.5X₀ - 0.5X₁. For two qubits we can also diagonalize the full matrix to get an exact reference.

In [ ]:
import torch
import flagquantum as fq
from flagquantum.algorithms import Hamiltonian, pauli_term
import matplotlib.pyplot as plt

H = Hamiltonian(
    [
        pauli_term(-1.0, "ZZ", (0, 1)),
        pauli_term(-0.5, "X", (0,)),
        pauli_term(-0.5, "X", (1,)),
    ]
)
exact_energy = torch.linalg.eigvalsh(H.matrix()).min().real.item()
print("Exact ground energy:", exact_energy)


## 2. Build a family of trial states

The adjustable circuit is called an ansatz. Its parameters choose a state; the loss is that state’s energy.

In [ ]:
def ansatz(theta):
    q = fq.Circuit(2)
    q.ry(0, theta[0])
    q.ry(1, theta[1])
    q.cx(0, 1)
    q.ry(0, theta[2])
    q.ry(1, theta[3])
    return q


torch.manual_seed(7)
theta = torch.nn.Parameter(0.2 * torch.randn(4))
optimizer = torch.optim.Adam([theta], lr=0.08)
print("Initial energy:", H.expectation(ansatz(theta)).item())


## 3. Minimize the energy

The exact answer is used only for checking, not by the optimizer. This method is the variational quantum eigensolver, or VQE.

In [ ]:
energies = []
for step in range(160):
    optimizer.zero_grad()
    energy = H.expectation(ansatz(theta)).sum()
    energy.backward()
    optimizer.step()
    energies.append(energy.item())
final_energy = H.expectation(ansatz(theta)).item()
print("Final energy:", final_energy, "Gap:", final_energy - exact_energy)
assert abs(final_energy - exact_energy) < 0.02
plt.plot(energies, label="VQE")
plt.axhline(exact_energy, color="black", linestyle="--", label="Exact")
plt.xlabel("Step")
plt.ylabel("Energy")
plt.legend()
plt.show()


## Make it yours

Change the transverse field from 0.5 to 0.1 or 1.5 in both X terms. Repeat from initialization. Remove one rotation layer: does the energy gap come from poor optimization or an ansatz that cannot represent the state?